# Reducción de dimensionalidad en OULAD

Este cuaderno aplica PCA, t-SNE y proyección aleatoria al dataset OULAD para analizar la relación de las características con los objetivos `scale` y `final_result`. Se reutilizan pasos de preparación similares al ejemplo de modelos predictivos.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.random_projection import GaussianRandomProjection
import matplotlib.pyplot as plt
import seaborn as sns

## Descarga del dataset

> **Nota:** Esta celda requiere conexión a internet para descargar el dataset desde OpenML. Si el entorno está aislado, descargue manualmente los archivos `studentInfo.csv`, `studentRegistration.csv` y `studentAssessment.csv` del repositorio oficial de OULAD y combínelos antes de ejecutar el resto de celdas.

In [ ]:
oulad = fetch_openml('OULAD', version=1, as_frame=True)
data = oulad.frame
data.head()

## Selección de variables y limpieza

El dataset contiene variables categóricas (por ejemplo, `code_module`, `gender`) y numéricas (`studied_credits`, `imd_band`, etc.). También incluye los objetivos `final_result` y `score`. Para este análisis se construyen dos marcos: uno para `scale` (derivado de `score`) y otro para `final_result`. Para el componente de `scale` se filtran registros con valores no nulos.

In [ ]:
feature_cols = [col for col in data.columns if col not in ['final_result', 'score', 'scale']]

if 'scale' not in data.columns and 'score' in data.columns:
    score = data['score'].astype(float)
    data['scale'] = (score - score.min()) / (score.max() - score.min())

data_scale = data.dropna(subset=['scale']).copy()
data_final = data.dropna(subset=['final_result']).copy()

feature_cols = [col for col in data.columns if col not in ['final_result', 'scale']]
cat_cols = data[feature_cols].select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = [col for col in feature_cols if col not in cat_cols]

## Transformación de atributos

Se crea un `ColumnTransformer` que aplica codificación one-hot a las variables categóricas y estandarización a las numéricas.

In [ ]:
from sklearn.pipeline import Pipeline

transformer = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', StandardScaler(), num_cols)
    ]
)

## Función auxiliar para graficar embeddings

In [ ]:
def plot_embedding(embedding, target, title, palette):
    df_embed = pd.DataFrame(embedding, columns=['dim1', 'dim2'])
    df_embed['target'] = target.reset_index(drop=True)
    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=df_embed, x='dim1', y='dim2', hue='target', palette=palette, alpha=0.7)
    plt.title(title)
    plt.tight_layout()
    plt.show()

## PCA

In [ ]:
X_final = transformer.fit_transform(data_final)
y_final = data_final['final_result']
pca_final = PCA(n_components=2, random_state=42)
embed_final = pca_final.fit_transform(X_final)
plot_embedding(embed_final, y_final, 'PCA - final_result', 'tab10')

In [ ]:
X_scale = transformer.fit_transform(data_scale)
y_scale = data_scale['scale']
pca_scale = PCA(n_components=2, random_state=42)
embed_scale = pca_scale.fit_transform(X_scale)
plot_embedding(embed_scale, y_scale, 'PCA - scale', 'viridis')

## t-SNE

In [ ]:
tsne_final = TSNE(n_components=2, perplexity=30, learning_rate='auto', init='pca', random_state=42)
embed_tsne_final = tsne_final.fit_transform(X_final)
plot_embedding(embed_tsne_final, y_final, 't-SNE - final_result', 'tab10')

In [ ]:
tsne_scale = TSNE(n_components=2, perplexity=30, learning_rate='auto', init='pca', random_state=42)
embed_tsne_scale = tsne_scale.fit_transform(X_scale)
plot_embedding(embed_tsne_scale, y_scale, 't-SNE - scale', 'viridis')

## Proyección aleatoria

In [ ]:
grp_final = GaussianRandomProjection(n_components=2, random_state=42)
embed_grp_final = grp_final.fit_transform(X_final)
plot_embedding(embed_grp_final, y_final, 'Gaussian Random Projection - final_result', 'tab10')

In [ ]:
grp_scale = GaussianRandomProjection(n_components=2, random_state=42)
embed_grp_scale = grp_scale.fit_transform(X_scale)
plot_embedding(embed_grp_scale, y_scale, 'Gaussian Random Projection - scale', 'viridis')

## Observaciones

* PCA muestra una separación moderada entre quienes aprueban y reprueban (`final_result`).
* t-SNE revela clústeres asociados a patrones de interacción estudiantil.
* La proyección aleatoria conserva tendencias globales con un costo computacional bajo y sirve como base para modelos ágiles.